# 

In [1]:
import os
import pandas as pd
import argparse
import torch as tc
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from tqdm.notebook import tqdm
from torchvision import models, datasets, transforms
from torchvision.transforms import v2
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from PIL import Image
from pathlib import Path

# Model loading

In [2]:
# Configuration

if tc.cuda.is_available():
    device = tc.device("cuda")
    print("Using GPU")
else:
    device = tc.device("cpu")
    print("Using CPU")

DIR_data = Path("/kaggle/input/aml-25-feathers-in-focus/data")

Using GPU


In [3]:
# Define functions

class ImageDFDataset(Dataset):
    def __init__(self, df, label_to_idx, transform=None):
        self.df = df.reset_index(drop=True)
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        rel_path = self.df.loc[idx, "image_path"]
        img_path = rel_path if os.path.isabs(rel_path) else os.path.join(DIR_data, rel_path)

        label_str = self.df.loc[idx, "label"]
        label = self.label_to_idx[label_str]

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

def build_label_mapping(df_train):
    classes = sorted(df_train["label"].unique())
    label_to_idx = {c: i for i, c in enumerate(classes)}
    idx_to_label = {i: c for c, i in label_to_idx.items()}
    return label_to_idx, idx_to_label

def build_resnet18(num_classes):
    model = models.resnet18(weights=None)   # NOT pretrained

    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(512, num_classes)
    )

    return model

def build_densenet121(num_classes):
    model = models.densenet121(weights=None)  # NOT pretrained

    model.classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(model.classifier.in_features, num_classes)
    )

    return model


# Transformations

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(128, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.5),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(144),
    transforms.CenterCrop(128),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize(144),
    transforms.CenterCrop(128),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [4]:
# Load data

df = pd.read_csv(DIR_data / "train_images.csv")
df["image_path"] = df["image_path"].str.replace("jpg", "png").str.replace("train_images", "train_images_rembg").str.lstrip("/")

test_df = pd.read_csv(DIR_data / "test_images_path.csv")
test_df["image_path"] = test_df["image_path"].str.replace("jpg", "png").str.replace("test_images", "test_images_rembg").str.lstrip("/")

# Split data

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# Save to CSV for error analysis
train_df.to_csv("train.csv")
val_df.to_csv("val.csv")

# Build label mapping

label_to_idx, idx_to_label = build_label_mapping(train_df)

# Build datasets and loaders

train_dataset = ImageDFDataset(train_df, label_to_idx, transform=train_transform)
val_dataset   = ImageDFDataset(val_df, label_to_idx, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory = True, persistent_workers = True)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory = True, persistent_workers = True)

test_dataset   = ImageDFDataset(test_df, label_to_idx, transform=test_transform)
test_loader   = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [6]:
# Train model

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BEST_CKPT_PATH = os.path.join(CHECKPOINT_DIR, "best.pt")
LAST_CKPT_PATH = os.path.join(CHECKPOINT_DIR, "last.pt")

def train_model(model, train_loader, val_loader, epochs=100, lr=1e-3):
    device = "cuda" if tc.cuda.is_available() else "cpu"
    model = model.to(device)
    print(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_loss = float("inf")

    epoch_bar = tqdm(range(epochs), desc="Epochs")

    for epoch in epoch_bar:
        model.train()
        running_loss = 0.0

        train_bar = tqdm(
            train_loader,
            desc=f"Train {epoch+1}/{epochs}",
            leave=False
        )

        for imgs, labels in train_bar:
            optimizer.zero_grad()

            imgs, labels_a, labels_b, lam = mixup(imgs, labels)
            imgs = imgs.to(device)
            labels_a = labels_a.to(device)
            labels_b = labels_b.to(device)

            outputs = model(imgs)
            loss = lam * criterion(outputs, labels_a) + (1 - lam) * criterion(outputs, labels_b)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            train_bar.set_postfix(loss=f"{loss.item():.4f}")

        val_loss = evaluate(model, val_loader, criterion, device)

        tc.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "best_loss": best_loss,
                "val_loss": val_loss,
            },
            LAST_CKPT_PATH
        )

        if val_loss < best_loss:
            best_loss = val_loss
            tc.save(
                {
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict(),
                    "best_loss": best_loss,
                    "val_loss": val_loss,
                },
                BEST_CKPT_PATH
            )
            tqdm.write(f"✅ New best model saved (val_loss={best_loss:.6f})")

        scheduler.step()

        epoch_bar.set_postfix(
            train_loss=f"{running_loss/len(train_loader):.4f}",
            val_loss=f"{val_loss:.4f}"
        )

    return model

def make_dataloaders(df_train, df_val, batch_size=64):

    label_to_idx, idx_to_label = build_label_mapping(df_train)
    num_classes = len(label_to_idx)

    train_dataset = ImageDFDataset(df_train, label_to_idx, transform=train_transform)
    val_dataset   = ImageDFDataset(df_val, label_to_idx, transform=val_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory = True, persistent_workers = True)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory = True, persistent_workers = True)

    return train_loader, val_loader, num_classes, label_to_idx, idx_to_label

def save_checkpoint(path, model, optimizer, epoch, best_val_loss, extra=None):
    ckpt = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "best_val_loss": best_val_loss,
    }
    if extra:
        ckpt.update(extra)
    tc.save(ckpt, path)

def load_checkpoint(path, model, optimizer=None, map_location="cpu"):
    ckpt = tc.load(path, map_location=map_location)
    model.load_state_dict(ckpt["model_state"])
    if optimizer is not None and "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
    return ckpt

def mixup(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size()[0]
    index = tc.randperm(batch_size)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

@tc.inference_mode()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item()

    return total_loss / len(loader)

model = build_densenet121(200) # Change for correct model

train_loader, val_loader, num_classes, label_to_idx, idx_to_label = make_dataloaders(train_df, val_df)

train_model(model, train_loader, val_loader, epochs=120, lr=1e-3)

cuda


Epochs:   0%|          | 0/120 [00:00<?, ?it/s]

Train 1/120:   0%|          | 0/50 [00:00<?, ?it/s]

✅ New best model saved (val_loss=10.162689)


Train 2/120:   0%|          | 0/50 [00:00<?, ?it/s]

...

Train 35/120:   0%|          | 0/50 [00:00<?, ?it/s]

Train 36/120:   0%|          | 0/50 [00:00<?, ?it/s]

Train 37/120:   0%|          | 0/50 [00:00<?, ?it/s]

Train 38/120:   0%|          | 0/50 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [64]:

# Load model

DIR_model = Path("/kaggle/input")

def build_resnet18(num_classes):
    model = models.resnet18(weights=None)   # NOT pretrained

    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(512, num_classes)
    )

    return model

model = build_resnet18(200)

model = model.to(device)

state_dict = tc.load(DIR_model / "birdclassifier/pytorch/default/1/best.pt")

model.load_state_dict(state_dict['model_state'])

model.eval()


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

# Feature extraction

In [ ]:
# Remove last classification layer of ResNet model

feature_extractor = nn.Sequential(*(list(model.children())[:-1]))

In [ ]:
# Load attributes

attributes = np.load(DIR_data / "attributes.npy")

attributes.shape

In [ ]:
# Extract features from training data

all_features = []
all_labels = []

with tc.no_grad():
    for data, labels in train_loader:
        
        data = data.to(device)
        
        features = feature_extractor(data)

        features_2d = features.reshape(features.size(0), -1).cpu().numpy()

        attribute_values = attributes[labels]

        attribute_values_2d = np.array(attribute_values).reshape(len(labels), -1)

        combined_features = np.concatenate((features_2d, attribute_values_2d), axis = 1)
        
        all_features.append(combined_features)
        all_labels.append(labels.cpu().numpy())
        
X_features = np.concatenate(all_features)
y_labels = np.concatenate(all_labels)
    
# Create dataframe

feature_cols = [f"feature_{i+1}" for i in range(X_features.shape[1])]

df_features_train = pd.DataFrame(X_features, columns = feature_cols)
df_features_train["label"] = y_labels

df_features_train

In [ ]:
# Extract features from validation data

all_features = []
all_labels = []

with tc.no_grad():
    for data, labels in val_loader:
        
        data = data.to(device)
        
        features = feature_extractor(data)

        features_2d = features.reshape(features.size(0), -1).cpu().numpy()

        attribute_values = attributes[labels]

        attribute_values_2d = np.array(attribute_values).reshape(len(labels), -1)

        combined_features = np.concatenate((features_2d, attribute_values_2d), axis = 1)
        
        all_features.append(combined_features)
        all_labels.append(labels.cpu().numpy())
        
X_features = np.concatenate(all_features)
y_labels = np.concatenate(all_labels)
    
# Create dataframe

feature_cols = [f"feature_{i+1}" for i in range(X_features.shape[1])]

df_features_val = pd.DataFrame(X_features, columns = feature_cols)
df_features_val["label"] = y_labels

df_features_val

# Random Forest Training

In [ ]:
df_features = pd.concat((df_features_train, df_features_val), axis=0)

df_features

In [ ]:
# Split data

X = df_features.drop("label", axis=1) 
y = df_features["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Define RF

rf_classifier = RandomForestClassifier(
    n_estimators=500, 
    random_state=42, 
    n_jobs=-1,
    max_depth=20
)

# Train classifier

rf_classifier.fit(X_train, y_train)

# Test Time Augmentation

In [65]:
class TestDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        rel_path = self.df.loc[idx, "image_path"].lstrip("/")
        img_path = os.path.join(DIR_data, rel_path)

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        img_id = self.df.loc[idx, "id"]
        return img, img_id

In [66]:
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

fivecrop_tf = transforms.Compose([
    transforms.Resize(144),
    transforms.FiveCrop(128),
    transforms.Lambda(lambda crops: tc.stack([
        transforms.Normalize(mean, std)(transforms.ToTensor()(c))
        for c in crops
    ]))
])

In [67]:
test_dataset = TestDataset(test_df, transform=fivecrop_tf)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [68]:
@tc.inference_mode()
def predict_tta_fivecrop(model, loader, device, do_flip=True):
    model.eval()
    all_ids = []
    all_pred_idx = []
    all_probs = []

    for crops, ids in tqdm(loader, desc = "Predicting..."):
        B, NC, C, H, W = crops.shape

        if tc.is_tensor(ids):
            ids_list = ids.detach().cpu().view(-1).tolist()
        else:
            ids_list = list(ids)

        crops = crops.view(B * NC, C, H, W).to(device)

        logits = model(crops)
        probs = tc.softmax(logits, dim=1)

        if do_flip:
            crops_f = tc.flip(crops, dims=[3])
            logits_f = model(crops_f)
            probs_f = tc.softmax(logits_f, dim=1)
            probs = (probs + probs_f) / 2.0

        probs = probs.view(B, NC, -1).mean(dim=1)
        pred_list = probs.argmax(dim=1).detach().cpu().tolist()

        max_probs, _ = tc.max(probs, dim=1)
        
        all_probs.extend(max_probs.detach().cpu().numpy().tolist())
            
        all_ids.extend(ids_list)
        all_pred_idx.extend(pred_list)

    return all_ids, all_pred_idx, all_probs

In [69]:
all_ids = []
pred_idx = []
all_probs = []

In [70]:
all_ids, pred_idx, all_probs = predict_tta_fivecrop(model, test_loader, device, do_flip=True)
print("ids:", len(all_ids), "preds:", len(pred_idx), "probs:", len(all_probs))
assert len(all_ids) == len(pred_idx)
assert len(all_ids) == len(all_probs)

Predicting...:   0%|          | 0/63 [00:00<?, ?it/s]

ids: 4000 preds: 4000 probs: 4000


In [71]:
pred_labels = [idx_to_label[i] for i in pred_idx]

preds_test_df = pd.DataFrame({
    "id": all_ids,
    "label": pred_labels,
    "prob" : all_probs,
})

In [72]:
low_conf_df = preds_test_df[preds_test_df["prob"] < 0.1]
low_conf_counts = low_conf_df["label"].value_counts()

low_conf_counts_filt = low_conf_counts[low_conf_counts > 5]

low_conf_counts_filt

label
31     16
33     14
78     12
22     12
43     11
4      10
37     10
11     10
36      9
17      8
91      8
13      8
115     7
76      7
40      7
39      7
100     7
20      7
7       7
128     7
52      6
32      6
18      6
130     6
118     6
68      6
Name: count, dtype: int64

In [73]:
total_count = len(preds_test_df)
low_conf_count = (preds_test_df["prob"] < 0.1).sum()

low_conf_perc = (low_conf_count/total_count) * 100
low_conf_perc

14.725

In [74]:
high_conf_df = preds_test_df[preds_test_df["prob"] > 0.75]
high_conf_counts = high_conf_df["label"].value_counts()

high_conf_counts_filt = high_conf_counts[high_conf_counts > 5]

high_conf_counts_filt

label
54    8
12    6
42    6
55    6
Name: count, dtype: int64

In [75]:
total_count = len(preds_test_df)
high_conf_count = (preds_test_df["prob"] > 0.75).sum()

high_conf_perc = (high_conf_count/total_count) * 100
high_conf_perc

3.95

# Error analysis

In [ ]:
val_df["id"] = val_df.index.to_series() + 1

val_dataset = TestDataset(val_df, transform=fivecrop_tf)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [ ]:
all_ids, pred_idx = predict_tta_fivecrop(model, val_loader, device, do_flip=True)
print("ids:", len(all_ids), "preds:", len(pred_idx))
assert len(all_ids) == len(pred_idx)

In [ ]:
pred_labels = [idx_to_label[i] for i in pred_idx]
true_labels = val_df["label"]

preds_val_df = pd.DataFrame({
    "id": all_ids,
    "true_label": true_labels,
    "pred_label": pred_labels
})

preds_val_df

In [ ]:
all_classes = np.unique(list(true_labels) + list(pred_labels))

cm = confusion_matrix(
    y_true=true_labels,
    y_pred=pred_labels,
    labels=all_classes
)

In [ ]:
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, )

fig, ax = plt.subplots(figsize=(200, 200))
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Confusion Matrix for Bird Classifier')
plt.show()

In [ ]:
bird_names = np.load(DIR_data / "class_names.npy", allow_pickle = True)

bird_names

In [ ]:
# Most confused pairs

cm_errors = cm.copy()

np.fill_diagonal(cm_errors, 0)

confused_pairs = []
num_classes = cm_errors.shape[0]

for i in range(num_classes):
    for j in range(num_classes):
        if cm[i, j] > 0:
            actual_id = all_classes[i]
            pred_id = all_classes[j]
            count = cm_errors[i, j] + cm_errors[j, i]
            confused_pairs.append((count, actual_id, pred_id))

confused_pairs.sort(key=lambda x: x[0], reverse=True)

top_3_pairs = confused_pairs[:10]

top_3_pairs

# 009.Brewer_Blackbird
# 027.Shiny_Cowbird

In [ ]:
train_df[train_df['label'] == 199]

In [ ]:
# Most correctly classified birds

correct_preds = np.diag(cm)

correct_counts = []
for i in range(len(correct_preds)):
    count = correct_preds[i]
    class_id = all_classes[i]
    correct_counts.append((count, class_id))

correct_counts.sort(key=lambda x: x[0], reverse=True)

top_3_correct = correct_counts[:10]

top_3_correct

# 

In [ ]:
# Most wrongly classified birds

wrong_counts = np.sum(cm_errors, axis=1)

wrong_counts = list(zip(wrong_counts, all_classes))

wrong_counts.sort(key=lambda x: x[0], reverse=True)

top_3_wrong = wrong_counts[:5]

top_3_wrong

# Brewer_Blackbird
# Yellow_bellied_Flycatcher

In [ ]:
cm.sum()

In [ ]:
np.sum(cm_errors, axis = 1)

In [ ]:
np.sum(cm, axis = 1)

In [ ]:
accuracy_score(true_labels, pred_labels)

In [127]:
counts = train_df['label'].value_counts()
print(counts)

label
2      28
1      28
4      28
9      27
10     27
       ..
194     5
197     4
198     4
200     4
199     4
Name: count, Length: 200, dtype: int64


In [130]:
counts_low = counts[counts < 10]
counts_low

label
158    9
160    9
159    9
161    9
162    9
163    9
157    9
169    8
165    8
167    8
164    8
166    8
168    8
170    8
175    7
174    7
173    7
176    7
172    7
171    7
182    6
177    6
183    6
185    6
190    6
184    6
179    6
180    6
189    6
186    6
187    6
181    6
178    6
188    6
193    5
192    5
196    5
191    5
195    5
194    5
197    4
198    4
200    4
199    4
Name: count, dtype: int64

In [136]:
recall = []
precision = []

for i, counts in counts_low.items():
    i = i - 1
    tp = cm[i, i]
    
    total_true = cm[i,:].sum()
    rec = tp / total_true
    rec_perc = rec * 100

    total_pred = cm[:, i].sum()
    if total_pred > 0:
        prec = tp / total_pred
    else:
        prec = 0
    prec_perc = prec * 100
    
    recall.append(rec_perc)
    precision.append(prec_perc)

avg_recall = sum(recall) / len(recall)

print(f"Recall {avg_recall}")

avg_prec = sum(precision) / len(precision)

print(f"Precision {avg_prec}")

Recall 38.63636363636363
Precision 39.3560606060606


In [138]:
counts = train_df['label'].value_counts()

counts_high = counts[counts > 20]
counts_high

label
2     28
1     28
4     28
9     27
10    27
20    26
17    26
11    26
23    26
19    26
14    26
15    26
3     26
13    26
16    26
28    25
25    25
29    25
27    25
26    25
21    25
30    25
12    25
22    25
34    24
35    24
33    24
31    24
37    24
36    24
41    23
39    23
42    23
38    23
43    23
44    23
40    23
54    22
50    22
24    22
56    22
49    22
47    22
53    22
48    22
55    22
46    22
52    22
32    22
7     22
51    22
57    22
45    22
61    21
63    21
60    21
62    21
59    21
64    21
58    21
Name: count, dtype: int64

In [139]:
recall = []
precision = []

for i, counts in counts_high.items():
    i = i - 1
    tp = cm[i, i]
    
    total_true = cm[i,:].sum()
    rec = tp / total_true
    rec_perc = rec * 100

    total_pred = cm[:, i].sum()
    if total_pred > 0:
        prec = tp / total_pred
    else:
        prec = 0
    prec_perc = prec * 100
    
    recall.append(rec_perc)
    precision.append(prec_perc)

avg_recall = sum(recall) / len(recall)

print(f"Recall {avg_recall}")

avg_prec = sum(precision) / len(precision)

print(f"Precision {avg_prec}")

Recall 54.57142857142858
Precision 53.65746753246753
